In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

print("Libraries Imported Successfully!")

In [ ]:
df = pd.read_csv("sales_data_sample.csv", encoding='latin1')

print("\nFirst 5 Records:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
print(df.info())

print("\nColumn Names:")
print(df.columns.tolist())

In [ ]:
print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

# Remove duplicate records
df = df.drop_duplicates()

print("\nDataset after removing duplicates:")
print(df.shape)

In [ ]:

# Convert order date into datetime format
df['ORDERDATE'] = pd.to_datetime(
    df['ORDERDATE'],
    errors='coerce'
)

# Convert sales into numeric format
df['SALES'] = pd.to_numeric(
    df['SALES'],
    errors='coerce'
)

# Remove records with missing important values
df = df.dropna(
    subset=['CUSTOMERNAME', 'ORDERDATE', 'SALES']
)

# Remove invalid sales values
df = df[df['SALES'] > 0]

print("\nCleaned Dataset:")
print(df.head())

In [ ]:
# Create customer-level RFM analysis

# Reference date for calculating recency
reference_date = df['ORDERDATE'].max() + pd.Timedelta(days=1)

# Group data customer-wise
customer_data = df.groupby('CUSTOMERNAME').agg({

    'ORDERDATE': lambda x: (reference_date - x.max()).days,

    'ORDERNUMBER': 'nunique',

    'SALES': 'sum'

})

# Rename columns
customer_data.columns = [
    'Recency',
    'Frequency',
    'Monetary'
]

print("\nCustomer RFM Data:")
print(customer_data.head())

print("\nTotal Customers:", len(customer_data))

In [ ]:
# ---------------- SECTION 6: FEATURE SELECTION ----------------

# Select RFM features for clustering
X = customer_data[
    ['Recency', 'Frequency', 'Monetary']
]

print("\nSelected Features:")
print(X.head())


In [ ]:
# ---------------- SECTION 7: FEATURE SCALING ----------------

# Apply log transformation to reduce skewness
X_log = np.log1p(X)

# Standardize features
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_log)

print("\nFeature Scaling Completed!")

print("Scaled Data Shape:", X_scaled.shape)


In [ ]:
# ---------------- SECTION 8: ELBOW METHOD ----------------

# Calculate WCSS for different cluster values

wcss = []

max_k = min(10, len(X_scaled))

for i in range(1, max_k + 1):

    kmeans = KMeans(
        n_clusters=i,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    wcss.append(kmeans.inertia_)


# Plot Elbow Graph
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, max_k + 1),
    wcss,
    marker='o'
)

plt.title("Elbow Method for Optimal Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.xticks(range(1, max_k + 1))
plt.grid(True)

plt.show()

In [ ]:
# ---------------- SECTION 9: APPLY K-MEANS CLUSTERING ----------------

# Select optimal number of clusters
# Examine the elbow graph and change K if required

optimal_k = 4

if optimal_k > len(X_scaled):
    optimal_k = len(X_scaled)

# Create K-Means model
kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

# Fit model and predict clusters
customer_data['Cluster'] = kmeans.fit_predict(X_scaled)

print("\nK-Means Clustering Completed!")

print("\nCustomer Segments:")
print(customer_data.head(20))


In [ ]:
# ---------------- SECTION 10: CLUSTER VISUALIZATION ----------------

plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=customer_data,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='viridis',
    s=100
)

plt.title("Customer Segmentation using K-Means")
plt.xlabel("Purchase Frequency")
plt.ylabel("Total Sales (Monetary)")
plt.legend(title="Customer Cluster")

plt.show()


In [ ]:

# ---------------- SECTION 11: CLUSTER ANALYSIS ----------------

# Calculate average RFM values for each cluster
cluster_analysis = customer_data.groupby('Cluster').agg({

    'Recency': 'mean',

    'Frequency': 'mean',

    'Monetary': 'mean'

}).round(2)

print("\n========== CUSTOMER CLUSTER ANALYSIS ==========")

print(cluster_analysis)

In [ ]:
# ---------------- SECTION 12: NUMBER OF CUSTOMERS IN EACH CLUSTER ----------------

cluster_counts = customer_data['Cluster'].value_counts().sort_index()

print("\nNumber of Customers in Each Cluster:")
print(cluster_counts)

plt.figure(figsize=(7, 4))

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.title("Number of Customers in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")

plt.show()



In [ ]:
# ---------------- SECTION 13: FINAL RESULTS ----------------

print("\n======================================")
print(" CUSTOMER SEGMENTATION USING K-MEANS")
print("======================================")

print("Algorithm Used       : K-Means Clustering")
print("Optimal Clusters Used:", optimal_k)
print("Total Customers      :", len(customer_data))

print("\nCluster Distribution:")
print(cluster_counts)

print("\nCluster-wise RFM Analysis:")
print(cluster_analysis)

print("\n======================================")
print("PROGRAM EXECUTED SUCCESSFULLY!")
print("======================================")


# ---------------- SECTION 14: SAVE RESULTS ----------------

customer_data.to_csv(
    "customer_segmentation_results.csv"
)

print("\nResults saved as customer_segmentation_results.csv")